# 04 · Adversarial attacks
PGD, CW, NES and counterfactual restoration on the object texture. Each writes snapshots + a loss history.

In [ ]:
import torch
from flynet_robustness.models import get_flynet
from flynet_robustness.decoders import load_decoder
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('FlyNet', device=device)
net = (model1, idxs1, model2, idxs2)          # FlyNet/CTRNN packing
decoder = load_decoder('FlyNet')
network_id = 'FlyNet'

## PGD ($L_2$)

In [ ]:
from flynet_robustness.attacks import run_pgd
out = run_pgd(net, decoder, network_id, norm='l2', eps=1000, steps=100,
              n_restarts=2, texttype='Black', device=device)
print('saved to', out)

## CW and NES

In [ ]:
from flynet_robustness.attacks import run_cw, run_nes
run_cw(net, decoder, network_id, norm='l2', c=100, steps=150, device=device)
run_nes(net, decoder, network_id, sigma=6, steps=100, n_samples=15, device=device)

## Counterfactual restoration

In [ ]:
from flynet_robustness.attacks import counterfactual_restoration
illusion, restored, h1, h2 = counterfactual_restoration(
    net, decoder, network_id, steps=250, texttype='Black', device=device)
from flynet_robustness.plotting import plot_texture
plot_texture(illusion); plot_texture(restored);

## CNN / RAFT targets
Load a feature extractor and pass it as `net`; ids: `vCNN_2_frames`, `vCNN_5_frames`, `gCNN_2_frames`, `RAFT_12`, ...

In [ ]:
from flynet_robustness.feature_extraction import load_cnn_network
# feat = load_cnn_network('vCNN_5_frames', device=device)   # n_frames handled by the registry
# dec  = load_decoder('vCNN_5_frames')
# run_pgd(feat, dec, 'vCNN_5_frames', norm='l2', eps=1000, device=device)